# Module 3 — Statistics & Probability
**FissionLab · AI/ML Foundations · Aarush**

Weeks 5 and 6:
- **W5 (Jun 29):** Descriptive stats, distributions, sampling
- **W6 (Jul 6):** Probability, correlation vs causation, train/val/test split

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

## Week 5 — Descriptive Statistics and Distributions

In [ ]:
np.random.seed(42)
# Simulate exam scores — approximately normal
scores = np.random.normal(loc=75, scale=12, size=200).clip(0, 100)

print('=== Descriptive Statistics ===')
print(f'n:       {len(scores)}')
print(f'Mean:    {scores.mean():.2f}')
print(f'Median:  {np.median(scores):.2f}')
print(f'Std:     {scores.std():.2f}')
print(f'Min/Max: {scores.min():.1f} / {scores.max():.1f}')
print(f'Q1/Q3:   {np.percentile(scores, 25):.1f} / {np.percentile(scores, 75):.1f}')

plt.figure(figsize=(10, 3))
plt.hist(scores, bins=30, color='steelblue', alpha=0.7, edgecolor='white')
plt.axvline(scores.mean(), color='#c9a84c', linewidth=2, label=f'Mean = {scores.mean():.1f}')
plt.axvline(np.median(scores), color='#ff6b6b', linewidth=2,
             linestyle='--', label=f'Median = {np.median(scores):.1f}')
plt.title('Score Distribution'); plt.xlabel('Score'); plt.ylabel('Count')
plt.legend(); plt.show()

### Exercise 3.1 — Skewed distribution
Create a right-skewed distribution (e.g., incomes) and observe how the mean and median differ.
Use `np.random.exponential(scale=50000, size=500)` for a skewed sample.

In [ ]:
incomes = np.random.exponential(scale=50000, size=500)

# TODO: compute mean and median; plot histogram with both marked
# Question: which is larger, mean or median? Why?

---
## Week 6 — Probability, Correlation, Train/Val/Test

In [ ]:
# Probability via simulation (law of large numbers)
n = 10000
# P(rolling a 6 on a fair die) = 1/6 ≈ 0.167
rolls = np.random.randint(1, 7, size=n)
p_six = (rolls == 6).mean()
print(f'P(roll=6) from {n} trials: {p_six:.4f}  (true: {1/6:.4f})')

# Conditional probability: P(pass | studied)
# Simulate: 60% of students study; given study → 90% pass; given no study → 40% pass
studied = np.random.rand(1000) < 0.6
p_pass = np.where(studied, np.random.rand(1000) < 0.9, np.random.rand(1000) < 0.4)
print(f'P(pass | studied)  = {p_pass[studied].mean():.3f}  (true: 0.9)')
print(f'P(pass | no study) = {p_pass[~studied].mean():.3f}  (true: 0.4)')
print(f'P(pass overall)    = {p_pass.mean():.3f}  (expected: 0.6*0.9 + 0.4*0.4 = {0.6*0.9+0.4*0.4:.3f})')

In [ ]:
# Correlation vs Causation demo
# Ice cream sales and drowning rates both correlate with temperature (the confounder)
np.random.seed(0)
temperature = np.random.normal(70, 15, 365)
ice_cream = 0.5 * temperature + np.random.normal(0, 5, 365)
drownings = 0.3 * temperature + np.random.normal(0, 3, 365)

print(f'Corr(ice_cream, drownings): {np.corrcoef(ice_cream, drownings)[0,1]:.3f}')
print(f'Corr(temperature, ice_cream): {np.corrcoef(temperature, ice_cream)[0,1]:.3f}')
print('High correlation between ice cream and drownings — but ice cream does NOT cause drownings!')
print('The confounder (temperature) drives both.')

In [ ]:
# Proper train/val/test split on the Breast Cancer dataset
data = load_breast_cancer()
X, y = data.data, data.target
print(f'Full dataset: {X.shape[0]} samples × {X.shape[1]} features')

# Step 1: carve out test set first (never touch until final evaluation)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
# Step 2: split remaining into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.15, random_state=42, stratify=y_trainval
)

print(f'Train:      {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Validation: {X_val.shape[0]} ({X_val.shape[0]/len(X)*100:.0f}%)')
print(f'Test:       {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.0f}%)')

# Class balance check (stratify ensures classes stay proportional)
print(f'Train positive rate:  {y_train.mean():.3f}')
print(f'Val positive rate:    {y_val.mean():.3f}')
print(f'Test positive rate:   {y_test.mean():.3f}')

### Overfitting Intuition
A model that memorizes training data will score perfectly on train but poorly on unseen data.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

train_accs, val_accs, depths = [], [], list(range(1, 20))

for depth in depths:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    train_accs.append(model.score(X_train, y_train))
    val_accs.append(model.score(X_val, y_val))

plt.figure(figsize=(10, 4))
plt.plot(depths, train_accs, 'o-', color='steelblue', label='Train accuracy')
plt.plot(depths, val_accs, 'o-', color='#ff6b6b', label='Validation accuracy')
plt.axvline(depths[np.argmax(val_accs)], color='#c9a84c', linestyle='--',
             label=f'Best val depth = {depths[np.argmax(val_accs)]}')
plt.title('Overfitting: Train vs Validation Accuracy by Tree Depth')
plt.xlabel('Max Depth'); plt.ylabel('Accuracy')
plt.legend(); plt.show()
print('Notice: train accuracy → 1.0 as depth → ∞, but val accuracy peaks then falls.')

---
## Self-Check
1. A dataset has mean=100 and median=65. Is this left-skewed or right-skewed? Explain.
2. Why do we use `stratify=y` in `train_test_split`?
3. In the tree depth plot, at approximately what depth does overfitting begin? How can you tell?

*Your answers here...*